# Dataset

Lista de puntajes obtenidos por diferentes jugadores oficialmente reconocidos por la Federación Internacional de Ajedrez. Los puntajes son mensuales y se cuenta con información de todos los meses del 2025. Se han considerado solo los puntajes del modo de juego "Blitz". Se descargaron 12 archivos XML con los puntajes, que luego fueron integrados en 1 solo archivo.

Fuente: https://ratings.fide.com/download_lists.phtml

In [1]:
import pandas as pd

df = pd.read_csv('data/combined.csv')

# Data profiling

## Estructura del dataset

Se cuenta con 14 variables y 3 149 829 de observaciones. Hay variables de tipo numérico y cadena de texto.

|Name|Type|Description|
|---|---|---|
|fideid|Int|identification number of a player within FIDE database|
|name|str|name of a player|
|country|str|Federation of a player|
|sex|str|sex of a player|
|title|str|title of a player|
|w_title|str|TBD|
|o_title|str|Other titles of a player|
|foa_title|str|TBD|
|rating|Int|Blitz Rating|
|games|Int|number of BLITZ rating games in given period|
|k|Int|BLITZ rating K factor|
|birthday|float|year of birth of a player|
|flag|str|flag of inactivity|
|filename|str|name of the file that holds the data for the month|



## Calidad del dataset

El dataset presenta buena calidad en sus variables principales (`rating`, `games`, `k`, `fideid`, `name`, `country`, `sex`). Los valores ausentes se concentran en columnas de títulos. La columna `flag` también presenta un alto porcentaje de NaN, que se trabajará en el pre-procesamiento ya que tiene un significado especial.

### Completitud

Las columnas de título (`title`, `w_title`, `o_title`, `foa_title`) presentan entre 93% y 99% de valores ausentes. Esto es esperado ya que solo los jugadores que han obtenido un título oficial de la FIDE los registran. La columna `flag` muestra un 34.64% de NaN, pero esto tampoco representa un problema de completitud ya que corresponde a información mal codificada. El único problema real de completitud es `birthday`, con 1.13% de registros (35,751) sin año de nacimiento.

In [4]:
# get count and percentage of missing data for each column
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
# create a dataframe to display the results
missing_df = pd.DataFrame({'Missing Count': missing_data, 'Missing Percentage': missing_percentage})
# drop columns with 0 missing rows
missing_df = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)


           Missing Count  Missing Percentage
title            2954110           93.786331
w_title          3110641           98.755838
o_title          3114808           98.888130
foa_title        3091700           98.154504
birthday           35751            1.135014
flag             1091180           34.642505


### Unicidad

El dataset es de tipo panel, donde cada fila representa a un jugador en un mes específico. Existen 283,291 jugadores únicos (`fideid`) que aparecen hasta 12 veces cada uno, pero sin repetirse en el mismo mes. La combinación `fideid` + `flag` presenta 2 808 353 coincidencias, lo que refleja que un mismo jugador mantiene el mismo estado de actividad durante varios meses consecutivos.

In [7]:
# count duplicates across all columnas, with exception of filename
duplicate_count = df.drop(columns=['filename']).duplicated().sum()
print(f'Number of duplicate rows (excluding filename): {duplicate_count}')

Number of duplicate rows (excluding filename): 0


In [11]:
# count unique fideid values
unique_fideid_count = df['fideid'].nunique()
print(f'Number of unique fideid values: {unique_fideid_count}')

Number of unique fideid values: 283291


### Validez

Se verificó que los valores de las variables numéricas y categóricas se encuentren dentro de los rangos esperados. El `rating` está entre 1400 y 2889 en el 100% de los registros, en línea con el piso oficial de la FIDE. Los valores de `k` son únicamente 10, 20 y 40 (los tres valores válidos del sistema Elo). Los valores de `sex` son exclusivamente 'M' y 'F', y ningún jugador masculino posee un título femenino (WGM, WIM, WFM, WCM). Resalta que en `birthday` 34 764 registros corresponden a años de nacimiento posteriores a 2015 (hasta 2021), lo que implicaría jugadores de entre 4 y 9 años de edad. Aunque inusual, podría tratarse de datos válidos.

In [ ]:
# Rating bounds
print(f"Rating range: {df['rating'].min()} - {df['rating'].max()}")
print(f"Ratings outside [1400, 3000]: {((df['rating'] < 1400) | (df['rating'] > 3000)).sum()}")

# K-factor valid values
print(f"\nK-factor unique values: {sorted(df['k'].unique())}")

# Sex values and cross-check with women's titles
print(f"\nSex unique values: {df['sex'].unique()}")
males_with_women_title = df[(df['sex'] == 'M') & (df['title'].isin(['WGM', 'WIM', 'WFM', 'WCM']))]
print(f"Males with women's title: {len(males_with_women_title)}")

# Invalid birth years
print("\nInvalid birth years (>2015):")
print(df[df['birthday'] > 2015]['birthday'].value_counts().sort_index().to_frame())

### Consistencia

Se analizó la consistencia longitudinal de los atributos de cada jugador a lo largo de los 12 meses. Se encontraron 905 jugadores con diferencias en su nombre entre meses, probablemente debido a registros manuales. Asimismo, 338 jugadores cambian de `country` entre meses y 76 jugadores presentan valores distintos en `sex` entre meses. No se tiene certeza de si estos cambios son permitidos por la FIDE.

In [6]:
# Name inconsistencies across months
name_per_player = df.groupby('fideid')['name'].nunique()
print(f"Players with >1 name across months: {(name_per_player > 1).sum()}")

# Country changes
country_per_player = df.groupby('fideid')['country'].nunique()
print(f"\nPlayers with >1 country across months: {(country_per_player > 1).sum()}")

# Sex inconsistencies
sex_per_player = df.groupby('fideid')['sex'].nunique()
print(f"\nPlayers with >1 sex value across months: {(sex_per_player > 1).sum()}")

Players with >1 name across months: 905

Players with >1 country across months: 338

Players with >1 sex value across months: 76


# Preprocesamiento

Se aplicaron cinco decisiones de limpieza sobre el dataset original:

1. Eliminar columna `Unnamed: 0`. Columna residual generada al exportar el CSV desde pandas. Duplica el índice sin aportar información.
2. Consolidar columnas de título. Se fusionan `title` y `w_title` en una sola columna `title`, tomando `title` como valor primario y `w_title` como alternativa cuando `title` es nulo. `o_title` (árbitros, entrenadores) y `foa_title` (arena online) corresponden a categorías distintas a los títulos de juego y no son relevantes para el análisis de ratings, por lo que se eliminan.
3. Extraer mes desde `filename`. La columna `filename` codifica el mes en formato `blitz_jan25frl_xml.xml`. Se extrae como entero (1–12) en una nueva columna `month` y se elimina `filename`. Sin esta variable el análisis temporal no es viable.
4. Recodificar `flag` como columna `active`. `flag` codifica simultáneamente el estado de actividad y el sexo del jugador (`i`=inactivo masculino, `w`=femenino activa, `wi`=femenino inactiva, NaN=masculino activo). Se extrae únicamente el componente de actividad como booleano (`active`=True si el jugador estuvo activo en ese período). La información de sexo ya está contenida en la columna `sex`, por lo que no se pierde información al eliminar `flag`.
5. Se mantienen las filas con valores perdidos en `birthday`, pero se renombra la columna a `birth_year`.

In [7]:
print("Estado inicial:")
print(f"  Shape: {df.shape}")
print(f"  Columnas: {list(df.columns)}")
print()
print(df.head(3).to_string())

Estado inicial:
  Shape: (3149830, 15)
  Columnas: ['Unnamed: 0', 'fideid', 'name', 'country', 'sex', 'title', 'w_title', 'o_title', 'foa_title', 'rating', 'games', 'k', 'birthday', 'flag', 'filename']

   Unnamed: 0    fideid                      name country sex title w_title o_title foa_title  rating  games   k  birthday flag                filename
0           0  10245154     A B M Jobair, Hossain     BAN   M   NaN     NaN     NaN       NaN    1748      0  20    1998.0  NaN  blitz_apr25frl_xml.xml
1           1  10207538          A E M, Doshtagir     BAN   M   NaN     NaN     NaN       NaN    1916      0  20    1974.0    i  blitz_apr25frl_xml.xml
2           2  10680810  A hamed Ashraf, Abdallah     EGY   M   NaN     NaN     NaN       NaN    1845      0  20    2001.0    i  blitz_apr25frl_xml.xml


In [ ]:
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
             'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

# Paso 1: drop Unnamed: 0
df = df.drop(columns=['Unnamed: 0'])

# Paso 2: coalesce title columns, drop w_title / o_title / foa_title
df['title'] = df['title'].fillna(df['w_title'])
df = df.drop(columns=['w_title', 'o_title', 'foa_title'])

# Paso 3: extract month from filename, drop filename
df['month'] = df['filename'].str.extract(r'blitz_([a-z]+)25')[0].map(month_map)
df = df.drop(columns=['filename'])

# Paso 4: flag → active boolean, drop flag
# active=True: flag is NaN (active male) or 'w' (active female)
# active=False: flag is 'i' (inactive male) or 'wi' (inactive female)
df['active'] = ~df['flag'].isin(['i', 'wi'])
df = df.drop(columns=['flag'])

# Paso 5: rename birthday to birth_year
df = df.rename(columns={'birthday': 'birth_year'})

In [9]:
print("Estado final:")
print(f"  Shape: {df.shape}")
print(f"  Columnas: {list(df.columns)}")
print()
print(df.head(3).to_string())

Estado final:
  Shape: (3149830, 11)
  Columnas: ['fideid', 'name', 'country', 'sex', 'title', 'rating', 'games', 'k', 'birthday', 'month', 'active']

     fideid                      name country sex title  rating  games   k  birthday  month  active
0  10245154     A B M Jobair, Hossain     BAN   M   NaN    1748      0  20    1998.0      4    True
1  10207538          A E M, Doshtagir     BAN   M   NaN    1916      0  20    1974.0      4   False
2  10680810  A hamed Ashraf, Abdallah     EGY   M   NaN    1845      0  20    2001.0      4   False
